In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 Telco Customer Churn - EDA & Baseline Model\n",
    "\n",
    "This notebook covers:\n",
    "1. Data loading and exploration\n",
    "2. Exploratory Data Analysis (EDA)\n",
    "3. Baseline model training\n",
    "4. Feature importance analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score\n",
    "\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "# Load data\n",
    "df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')\n",
    "print(f"Dataset shape: {df.shape}")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Data Overview"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic info\n",
    "print("Data Types:\n", df.dtypes)\n",
    "print("\nMissing Values:\n", df.isnull().sum())\n",
    "print("\nDuplicated Rows:", df.duplicated().sum())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Target Distribution"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Churn distribution\n",
    "churn_counts = df['Churn'].value_counts()\n",
    "churn_pct = df['Churn'].value_counts(normalize=True) * 100\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# Count plot\n",
    "sns.countplot(data=df, x='Churn', ax=ax1, palette=['#2ecc71', '#e74c3c'])\n",
    "ax1.set_title('Churn Distribution (Count)', fontsize=14)\n",
    "ax1.set_ylabel('Number of Customers')\n",
    "for i, v in enumerate(churn_counts.values):\n",
    "    ax1.text(i, v + 50, str(v), ha='center', fontsize=12, fontweight='bold')\n",
    "\n",
    "# Pie chart\n",
    "colors = ['#2ecc71', '#e74c3c']\n",
    "ax2.pie(churn_pct, labels=['No Churn', 'Churn'], autopct='%1.1f%%', \
",
    "        colors=colors, startangle=90, explode=(0, 0.05))\n",
    "ax2.set_title('Churn Distribution (%)', fontsize=14)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f"Churn Rate: {churn_pct['Yes']:.1f}%")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Numerical Features Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Convert TotalCharges to numeric\n",
    "df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')\n",
    "\n",
    "numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "\n",
    "for idx, col in enumerate(numerical_cols):\n",
    "    # Distribution\n",
    "    sns.histplot(data=df, x=col, hue='Churn', kde=True, ax=axes[0, idx], palette=['#2ecc71', '#e74c3c'])\n",
    "    axes[0, idx].set_title(f'{col} Distribution')\n",
    "    \n",
    "    # Box plot\n",
    "    sns.boxplot(data=df, x='Churn', y=col, ax=axes[1, idx], palette=['#2ecc71', '#e74c3c'])\n",
    "    axes[1, idx].set_title(f'{col} by Churn')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Categorical Features Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Key categorical features\n",
    "cat_features = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for idx, col in enumerate(cat_features):\n",
    "    churn_by_cat = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)\n",
    "    \n",
    "    churn_by_cat.plot(kind='bar', ax=axes[idx], color='#e74c3c', alpha=0.8)\n",
    "    axes[idx].set_title(f'Churn Rate by {col}', fontsize=12)\n",
    "    axes[idx].set_ylabel('Churn Rate (%)')\n",
    "    axes[idx].tick_params(axis='x', rotation=45)\n",
    "    \n",
    "    # Add value labels\n",
    "    for i, v in enumerate(churn_by_cat.values):\n",
    "        axes[idx].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Encode categorical for correlation\n",
    "df_encoded = df.copy()\n",
    "for col in df_encoded.select_dtypes(include=['object']).columns:\n",
    "    if col != 'customerID':\n",
    "        df_encoded[col] = pd.Categorical(df_encoded[col]).codes\n",
    "\n",
    "# Correlation matrix\n",
    "corr_matrix = df_encoded.corr()\n",
    "\n",
    "plt.figure(figsize=(14, 12))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0)\n",
    "plt.title('Feature Correlation Matrix', fontsize=16)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Baseline Model"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Prepare data for baseline\n",
    "from sklearn.preprocessing import LabelEncoder, StandardScaler\n",
    "\n",
    "# Drop ID and target\n",
    "X = df.drop(['customerID', 'Churn'], axis=1)\n",
    "y = df['Churn'].map({'Yes': 1, 'No': 0})\n",
    "\n",
    "# Handle TotalCharges\n",
    "X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')\n",
    "X['TotalCharges'].fillna(X['MonthlyCharges'] * X['tenure'], inplace=True)\n",
    "\n",
    "# Encode categoricals\n",
    "categorical_cols = X.select_dtypes(include=['object']).columns\n",
    "for col in categorical_cols:\n",
    "    le = LabelEncoder()\n",
    "    X[col] = le.fit_transform(X[col])\n",
    "\n",
    "# Split\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X, y, test_size=0.2, random_state=42, stratify=y\n",
    ")\n",
    "\n",
    "# Scale\n",
    "scaler = StandardScaler()\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_test_scaled = scaler.transform(X_test)\n",
    "\n",
    "# Train baseline\n",
    "rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')\n",
    "rf.fit(X_train_scaled, y_train)\n",
    "\n",
    "# Evaluate\n",
    "y_pred = rf.predict(X_test_scaled)\n",
    "y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]\n",
    "\n",
    "print("Baseline Model Performance:\n")\n",
    "print(classification_report(y_test, y_pred))\n",
    "print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Feature Importance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature importance\n",
    "feature_importance = pd.DataFrame({\n",
    "    'feature': X.columns,\n",
    "    'importance': rf.feature_importances_\n",
    "}).sort_values('importance', ascending=False)\n",
    "\n",
    "plt.figure(figsize=(12, 8))\n",
    "sns.barplot(data=feature_importance.head(15), x='importance', y='feature', palette='viridis')\n",
    "plt.title('Top 15 Feature Importances', fontsize=14)\n",
    "plt.xlabel('Importance')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print("Top 10 Important Features:\n")\n",
    "print(feature_importance.head(10))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Key Insights Summary\n",
    "\n",
    "1. **Churn Rate**: ~26.5% of customers churn\n",
    "2. **Contract Type**: Month-to-month contracts have highest churn\n",
    "3. **Tenure**: New customers (low tenure) are more likely to churn\n",
    "4. **Payment Method**: Electronic check users churn more\n",
    "5. **Internet Service**: Fiber optic users have higher churn\n",
    "6. **Top Predictors**: Contract, tenure, MonthlyCharges, TotalCharges"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}